# 🎯 Activity Recognition - Pipeline de Processamento IoT

## 📊 Contexto

Pipeline de processamento de dados de **acelerômetros e giroscópios** coletados de smartphones e smartwatches para reconhecimento de atividades humanas.

**Dataset:** Heterogeneity Activity Recognition (UCI ML Repository)  
**Volume:** ~3GB (33.7M registros)  
**Dispositivos:** Phones e Watches  
**Sensores:** Accelerometer e Gyroscope

## 🏗️ Arquitetura Medallion

* 🟤 **Bronze** (`workspace.ing`) - Dados brutos dos CSVs
* 🟡 **Silver** (`workspace.wks`) - Dados limpos e enriquecidos com timestamps
* 🟢 **Gold** (`workspace.ucs`) - Agregações por usuário

---

## 🟤 Bloco 1: Setup - Importações e Schema

In [0]:
# ============================================================
# PASSO 1: IMPORTAÇÕES E DEFINIÇÃO DO SCHEMA
# ============================================================

from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
from pyspark.sql.functions import (
    col, lit, count, sum as spark_sum, countDistinct, collect_set,
    from_unixtime, lag, unix_timestamp, round as spark_round, to_date
)
from pyspark.sql.window import Window

print("✅ Importações realizadas com sucesso!\n")

# ============================================================
# DEFINIÇÃO DO SCHEMA EXPLÍCITO 
# ============================================================

# Schema para todos os arquivos de sensores
# Estrutura: Index, Arrival_Time, Creation_Time, x, y, z, User, Model, Device, gt
sensor_schema = StructType([
    StructField("Index", LongType(), True),
    StructField("Arrival_Time", LongType(), True),       # Timestamp em milissegundos
    StructField("Creation_Time", LongType(), True),      # Timestamp em nanossegundos
    StructField("x", DoubleType(), True),                # Eixo X do sensor
    StructField("y", DoubleType(), True),                # Eixo Y do sensor
    StructField("z", DoubleType(), True),                # Eixo Z do sensor
    StructField("User", StringType(), True),             # ID do usuário (a-i)
    StructField("Model", StringType(), True),            # Modelo do dispositivo
    StructField("Device", StringType(), True),           # ID único do dispositivo
    StructField("gt", StringType(), True)                # Ground Truth (atividade)
])

print("✅ Schema explícito definido:")
print(sensor_schema.simpleString())
print(f"\n📊 Total de campos: {len(sensor_schema.fields)}")

In [0]:
# ============================================================
# BLOCO 2: INGESTÃO BRONZE
# ============================================================

# Caminho base do volume
volume_base = "/Volumes/workspace/default/activity_recognition_data"

# Carregar 4 arquivos CSV com schema explícito
df_phones_accel = spark.read \
    .schema(sensor_schema) \
    .option("header", "true") \
    .csv(f"{volume_base}/Phones_accelerometer.csv") \
    .withColumn("sensor_type", lit("accelerometer")) \
    .withColumn("device_type", lit("phone"))

df_phones_gyro = spark.read \
    .schema(sensor_schema) \
    .option("header", "true") \
    .csv(f"{volume_base}/Phones_gyroscope.csv") \
    .withColumn("sensor_type", lit("gyroscope")) \
    .withColumn("device_type", lit("phone"))

df_watch_accel = spark.read \
    .schema(sensor_schema) \
    .option("header", "true") \
    .csv(f"{volume_base}/Watch_accelerometer.csv") \
    .withColumn("sensor_type", lit("accelerometer")) \
    .withColumn("device_type", lit("watch"))

df_watch_gyro = spark.read \
    .schema(sensor_schema) \
    .option("header", "true") \
    .csv(f"{volume_base}/Watch_gyroscope.csv") \
    .withColumn("sensor_type", lit("gyroscope")) \
    .withColumn("device_type", lit("watch"))

# Unificar em um único DataFrame (Bronze Layer)
df_raw = df_phones_accel \
    .unionByName(df_phones_gyro) \
    .unionByName(df_watch_accel) \
    .unionByName(df_watch_gyro)

print(f"✅ Bronze: {df_raw.count():,} registros unificados")

In [0]:
# ============================================================
# BLOCO 3A: SILVER LAYER - TRANSFORMAÇÕES
# ============================================================

# Conversão de timestamps para formato legível
# Arrival_Time: milissegundos -> segundos
# Creation_Time: nanossegundos -> segundos
df_with_timestamps = df_raw \
    .withColumn("arrival_timestamp", from_unixtime(col("Arrival_Time") / 1000)) \
    .withColumn("creation_timestamp", from_unixtime(col("Creation_Time") / 1000000000)) \
    .withColumn("date", to_date(col("arrival_timestamp")))

# Cálculo de intervalo temporal entre eventos consecutivos
# NOTA: Window particionada por User e Device (sem sensor_type - ver Notas de Qualidade)
window_spec = Window.partitionBy("User", "Device").orderBy("Arrival_Time")

df_silver = df_with_timestamps \
    .withColumn("prev_arrival_time", lag("Arrival_Time", 1).over(window_spec)) \
    .withColumn(
        "time_interval_ms",
        col("Arrival_Time") - col("prev_arrival_time")
    ) \
    .withColumn(
        "time_interval_sec",
        spark_round(col("time_interval_ms") / 1000, 3)
    ) \
    .drop("prev_arrival_time")

print(f"✅ Silver: {df_silver.count():,} registros processados")

In [0]:
# ============================================================
# BLOCO 3B: GOLD LAYER - AGREGAÇÕES
# ============================================================

# Agregações por usuário: métricas de negócio
df_gold_usuarios = df_silver.groupBy("User").agg(
    count("*").alias("total_registros"),
    countDistinct("Model").alias("qtd_modelos_distintos"),
    collect_set("Model").alias("modelos_operados"),
    countDistinct("Device").alias("qtd_dispositivos_distintos"),
    countDistinct("sensor_type").alias("qtd_tipos_sensores"),
    countDistinct("device_type").alias("qtd_tipos_dispositivos"),
    countDistinct("gt").alias("qtd_atividades_distintas")
).orderBy("User")

print(f"✅ Gold: {df_gold_usuarios.count():,} usuários agregados")
display(df_gold_usuarios)

In [0]:
# ============================================================
# BLOCO 4: OTIMIZAÇÃO E PERSISTÊNCIA
# ============================================================

catalog = "workspace"
schema_silver = "wks"
schema_gold = "ucs"

# Criar schemas se não existirem
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}")

# ============================================================
# SALVAR SILVER LAYER
# ============================================================
table_silver = f"{catalog}.{schema_silver}.activity_recognition_silver"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date", "device_type") \
    .saveAsTable(table_silver)

print(f"✅ Tabela Silver criada: {table_silver}")
print(f"   - Registros: {spark.table(table_silver).count():,}")
print(f"   - Particionamento: date, device_type")

# Otimizar com Z-ORDER
spark.sql(f"OPTIMIZE {table_silver} ZORDER BY (User, sensor_type)")
print(f"   - Otimização: ZORDER BY (User, sensor_type)")

# ============================================================
# SALVAR GOLD LAYER
# ============================================================
table_gold = f"{catalog}.{schema_gold}.activity_recognition_gold_usuarios"

df_gold_usuarios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_gold)

print(f"\n✅ Tabela Gold criada: {table_gold}")
print(f"   - Registros: {spark.table(table_gold).count():,}")

# Otimizar
spark.sql(f"OPTIMIZE {table_gold}")
print(f"   - Otimização: OPTIMIZE aplicado")

# ============================================================
# VACUUM (Gestão do histórico de versões)
# ============================================================
spark.sql(f"VACUUM {table_silver} RETAIN 168 HOURS")
spark.sql(f"VACUUM {table_gold} RETAIN 168 HOURS")
print(f"\n✅ VACUUM executado: 168 horas de retenção")

print(f"\n\n🎉 PIPELINE CONCLUÍDO COM SUCESSO!\n")
print(f"📊 Consulte as tabelas:")
print(f"   - {table_silver}")
print(f"   - {table_gold}")

# 🚧 Notas de Qualidade de Dados e Próximos Passos

## ⚠️ Problemas Identificados no Dataset

### 1️⃣ **String 'null' vs NULL Real (13.76% dos dados)**
* **Problema:** 4.643.613 registros contêm texto literal "null" no campo `gt` (não tipo NULL)
* **Impacto:** `.isNull()` retorna False - análise inicial reportou "sem nulos" (falso negativo)
* **Remediação:** Converter para NULL ou categoria 'unlabeled' na Silver Layer

### 2️⃣ **Duplicatas Massivas (99.36% - CRÍTICO)**
* **Problema:** 33.527.200 registros duplicados detectados
* **Impacto:** Campo `Index` não é único por combinação (Index, Device, sensor_type) - todas agregações infladas ~10x
* **Remediação:** Implementar `.dropDuplicates()` ou Window Functions com `row_number()` na Bronze Layer

### 3️⃣ **Escalas de Timestamp Inconsistentes (63%)**
* **Problema:** 21.259.512 registros com `Creation_Time` em milissegundos (esperado: nanossegundos)
* **Impacto:** Conversão `/1000000000` incorreta para 63% dos dados; 29% dos registros com Arrival ANTES de Creation (anomalia de sincronização)
* **Remediação:** Auto-detectar escala com `when(col < 1e18)` antes da conversão

### 4️⃣ **Window Function Incorreta**
* **Problema:** Window particionada por `(User, Device)` SEM `sensor_type`
* **Impacto:** Calcula intervalos entre acelerômetro e giroscópio (incorreto) - 84 combinações User+Device afetadas
* **Remediação:** Adicionar `sensor_type` ao `Window.partitionBy()`
```python
window_spec = Window.partitionBy("User", "Device", "sensor_type").orderBy("Arrival_Time")
```

### 5️⃣ **Outliers em Valores de Sensores (OK)**
* **Status:** Valores dentro do range físico esperado (-40 a +40 m/s² para acelerômetros)
* **Remediação:** Opcional - detecção baseada em IQR para casos extremos

### 6️⃣ **Desbalanceamento de Classes (Moderado)**
* **Status:** walk (5.580.900 registros) vs stairsdown (4.204.346 registros) - ratio 1.33x
* **Remediação:** Aplicar class weighting em modelos de ML (opcional)

---

## 🔧 Próximas Iterações

* 🧹 **Deduplicação obrigatória** na Bronze Layer antes de agregações
* ⏱️ **Correção de timestamps heterogêneos** com detecção automática de escala
* 🎯 **Ajuste da Window Function** com `sensor_type` na partição
* 🧑‍🔬 **Tratamento de registros 'null'** - converter para categoria válida ou remover
* 📊 **Feature Engineering** para modelos de ML - estatísticas de janela deslizante, FFT, etc.